# Phase 1: Triage and Routing

I build an explainable decision prototype using synthetic fixtures and the exact SVR Contracts v1.0 modules. I calculate priority using Hazem's approved text-only formula. I route only to the existing fictional demo agents.

Contents:
1. Setup, official contract loading, and shared helpers
2. Priority decision table
3. Automated synthetic tests
4. Ticket handoff and capacity lifecycle
5. Findings and remaining decisions

I do not read datasets, run models, or evaluate routing agreement. Arafa owns ticket assembly. This notebook is a prototype for review, not a production routing policy.


## 1. Setup and shared helpers

I pin the contract source to commit `6f21d9bd20af4a175fd9ac011ac89384f32e4c97`, originally inspected through `origin/team/arafa-systems-demo`. I load unchanged Git blobs into a private in-memory package so the empty local `src/svr/` stays untouched. Source SHA256 values and environment versions are printed below. There are no simplified replacement schemas.

I use the official `TextAnalysis`, optional `VoiceEvidence`, `TriageDecision`, agent reservation functions, ticket assembler, and store. The prototype revalidates inputs before reserving capacity and rejects mismatched voice IDs. It does not fuse voice evidence or infer urgency from state. Missing labels remain null and retain the text review flag.

I reserve once through `agents.assign_agent`. The successful caller owns that reservation until official store completion or an explicit standalone release. A construction failure rolls back the reservation. I keep fixture changes private to this notebook's package.


In [1]:
from __future__ import annotations

import copy
import hashlib
import io
import json
import platform
import subprocess
import sys
import types
import unittest
from concurrent.futures import ThreadPoolExecutor
from itertools import product
from pathlib import Path
from unittest.mock import patch

import pydantic
from pydantic import ValidationError

CONTRACT_REF = "origin/team/arafa-systems-demo"
CONTRACT_COMMIT = "6f21d9bd20af4a175fd9ac011ac89384f32e4c97"
CONFIG_VERSION = "ibthal-phase1-demo-0.2"
EXPECTED_BRANCH = "team/ibthal-triage-routing"
LABEL_FIELDS = (
    "complaint_category", "complaint_urgency", "customer_state_text", "churn_risk"
)
EXPECTED_DECISION_FIELDS = {
    "voicemail_id", "priority_score", "queue_priority", "assigned_agent_id",
    "assignment_status", "route_reason", "contributing_signals", "config_version",
    "evaluation", "schema_version", "label_version"
}

# I locate only the repository, never a dataset or customer-content directory.
ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / ".git").exists() and (p / "docs/project/roles-and-workflow.md").exists()),
    None,
)
if ROOT is None:
    raise RuntimeError("Run from the repository root or its notebook directory.")

def git_read(*args: str) -> bytes:
    """Read local Git objects only. No fetch, checkout, or worktree writes."""
    return subprocess.run(
        ["git", "-C", str(ROOT), *args], check=True, capture_output=True
    ).stdout

branch = git_read("branch", "--show-current").decode().strip()
if branch != EXPECTED_BRANCH:
    raise RuntimeError(f"Expected {EXPECTED_BRANCH}, found {branch}; no branch is changed.")
ref_commit = git_read("rev-parse", "--verify", CONTRACT_REF).decode().strip()
git_read("cat-file", "-e", CONTRACT_COMMIT + "^{commit}")
if not hasattr(pydantic.BaseModel, "model_validate"):
    raise RuntimeError("Official schemas require Pydantic 2. No substitute schema is used.")

# I use the exact Git blobs in a private package. I intentionally bypass svr's
# __init__, which imports the full pipeline. No source is copied to src/.
PACKAGE = "_ibthal_phase1_official"
for name in list(sys.modules):
    if name == PACKAGE or name.startswith(PACKAGE + "."):
        del sys.modules[name]

for suffix in ("", ".stages"):
    name = PACKAGE + suffix
    module = types.ModuleType(name)
    module.__package__ = name
    module.__path__ = []
    sys.modules[name] = module
    if suffix:
        setattr(sys.modules[PACKAGE], "stages", module)

SOURCE_MANIFEST = {}
def load_official(relative_name: str):
    path = "src/svr/" + relative_name.replace(".", "/") + ".py"
    source = git_read("show", f"{CONTRACT_COMMIT}:{path}")
    SOURCE_MANIFEST[path] = hashlib.sha256(source).hexdigest()
    name = PACKAGE + "." + relative_name
    module = types.ModuleType(name)
    module.__file__ = f"git:{CONTRACT_COMMIT}:{path}"
    module.__package__ = name.rpartition(".")[0]
    sys.modules[name] = module
    parent, _, child = name.rpartition(".")
    setattr(sys.modules[parent], child, module)
    exec(compile(source, module.__file__, "exec"), module.__dict__)
    return module

schemas = load_official("schemas")
agents = load_official("agents")
fixtures = load_official("fixtures")
official_triage = load_official("stages.triage")
ticket_stage = load_official("stages.ticket")
# These official stdlib/Pydantic modules support the store lifecycle test.
for name in ("latency", "stages.intake", "stages.stt",
             "stages.classification", "stages.voice"):
    load_official(name)
pipeline = load_official("pipeline")
store_module = load_official("store")

TextAnalysis = schemas.TextAnalysis
VoiceEvidence = schemas.VoiceEvidence
TriageDecision = schemas.TriageDecision
AssignmentStatus = schemas.AssignmentStatus
ComplaintCategory = schemas.ComplaintCategory
ComplaintUrgency = schemas.ComplaintUrgency
CustomerState = schemas.CustomerState
ChurnRisk = schemas.ChurnRisk
TicketStatus = schemas.TicketStatus

assert schemas.SCHEMA_VERSION == 1
assert schemas.LABEL_VERSION == "taxonomy-2026-11-15"
assert len(ComplaintCategory) == 11
assert set(TriageDecision.model_fields) == EXPECTED_DECISION_FIELDS

def synthetic_text(**changes) -> TextAnalysis:
    payload = copy.deepcopy(fixtures.FIXTURE_TEXT_ANALYSIS)
    payload.update(changes)
    return TextAnalysis.model_validate(payload)

def synthetic_voice(**changes) -> VoiceEvidence:
    payload = {
        "voicemail_id": fixtures.FIXTURE_VOICEMAIL_ID,
        "voice_state_candidate": "calm",
        "voice_confidence": 0.8,
        "model_version": "synthetic-voice-0.1",
    }
    payload.update(changes)
    return VoiceEvidence.model_validate(payload)

def synthetic_audio(voicemail_id: str):
    # No audio is opened, persisted, transcribed, or claimed to be measured.
    return schemas.AudioEvidence(
        voicemail_id=voicemail_id, format="wav",
        transcript=fixtures.FIXTURE_TRANSCRIPT,
        stt_provider="synthetic", stt_model_version="synthetic-stt-0.1",
    )

def load_snapshot() -> dict[str, int]:
    return {agent.id: agent.current_load for agent in agents.list_agents()}

INITIAL_LOADS = load_snapshot()

URGENCY_POINTS = {ComplaintUrgency.HIGH: 3, ComplaintUrgency.MEDIUM: 2, ComplaintUrgency.LOW: 1}
CHURN_POINTS = {ChurnRisk.AT_RISK: 2, ChurnRisk.NOT_AT_RISK: 0}
STATE_POINTS = {state: int(state in (CustomerState.MAD, CustomerState.FRUSTRATED))
                for state in CustomerState}

def calculate_priority(text: TextAnalysis):
    """Validate official text and score complete signals using Hazem's policy."""
    text = TextAnalysis.model_validate(text.model_dump())
    required = ("complaint_urgency", "churn_risk", "customer_state_text")
    missing = [field for field in required if getattr(text, field) is None]
    if missing:
        explanation = (
            "Priority inputs missing: " + ", ".join(missing)
            + ". Priority remains null for human review; missing-input policy requires Hazem confirmation."
        )
        return None, None, ["priority_missing=" + ",".join(missing)], explanation
    urgency = URGENCY_POINTS[text.complaint_urgency]
    churn = CHURN_POINTS[text.churn_risk]
    state = STATE_POINTS[text.customer_state_text]
    score = urgency + churn + state
    bucket = {1: "low", 2: "low", 3: "normal", 4: "normal", 5: "high", 6: "urgent"}[score]
    signals = [f"urgency_points={urgency}", f"churn_points={churn}",
               f"customer_state_points={state}", f"priority_score={score}",
               f"queue_priority={bucket}"]
    explanation = f"Priority calculation: {urgency} + {churn} + {state} = {score}; queue_priority={bucket}."
    return score, schemas.QueuePriority(bucket), signals, explanation

def triage_phase1(
    text: TextAnalysis, voice_evidence: VoiceEvidence | None = None
) -> TriageDecision:
    """Score approved text signals and reserve once through official demo routing."""
    if not isinstance(text, TextAnalysis):
        raise TypeError("text must be an official TextAnalysis object.")
    # Revalidate mutable objects before any reservation; model_copy skips validation.
    text = TextAnalysis.model_validate(text.model_dump())
    if voice_evidence is not None:
        if not isinstance(voice_evidence, VoiceEvidence):
            raise TypeError("voice_evidence must be an official VoiceEvidence object.")
        voice_evidence = VoiceEvidence.model_validate(voice_evidence.model_dump())
        if voice_evidence.voicemail_id != text.voicemail_id:
            raise ValueError("Text and voice evidence must belong to the same voicemail.")

    signals = [
        f"{field}={getattr(text, field).value if getattr(text, field) is not None else 'null'}"
        for field in LABEL_FIELDS
    ]
    signals.append(f"needs_human_review={str(text.needs_human_review).lower()}")
    if voice_evidence is not None:
        signals.append("voice_evidence=ignored_fusion_disabled")

    score, bucket, priority_signals, priority_reason = calculate_priority(text)
    signals.extend(priority_signals)
    agent = None
    try:
        if text.complaint_category is None:
            reason = (
                "Complaint category is missing; no skill is guessed. "
                "The decision stays unassigned for human routing."
            )
        else:
            # This function selects AND reserves under the official module's lock.
            # I do not call consume(), reimplement selection, or apply tier policy.
            agent = agents.assign_agent(text.complaint_category)
            if agent is None:
                reason = (
                    "No fictional seeded demo agent has a matching listed skill, "
                    "is on shift, and has spare capacity. The decision stays "
                    "unassigned for human routing."
                )
            else:
                reason = (
                    f"Fictional seeded demo routing assigns {agent.id}: its listed "
                    f"skill matches {text.complaint_category.value}, it is on shift, "
                    "and one capacity unit is reserved. Selection uses the existing "
                    "demo seed order, not an approved production tie-breaking policy."
                )
        if text.needs_human_review:
            reason += " Text analysis still requires human review; routing does not clear it."
        reason += " " + priority_reason + " Voice fusion is disabled."

        return TriageDecision(
            voicemail_id=text.voicemail_id,
            priority_score=score, queue_priority=bucket,
            assigned_agent_id=agent.id if agent else None,
            assignment_status=AssignmentStatus.ASSIGNED if agent else AssignmentStatus.UNASSIGNED,
            route_reason=reason, contributing_signals=signals,
            config_version=CONFIG_VERSION, evaluation=None,
            schema_version=schemas.SCHEMA_VERSION, label_version=schemas.LABEL_VERSION,
        )
    except Exception:
        # A failed decision never hands off its reservation to the pipeline.
        if agent is not None:
            agents.release(agent.id)
        raise

def handoff_arguments(text, decision, voice_evidence=None):
    """Synthetic keyword arguments for Arafa's assembler, not a new assembler."""
    return {
        "ticket_id": "ticket-synthetic-001",
        "voicemail_id": text.voicemail_id,
        "audio": synthetic_audio(text.voicemail_id),
        "text": text,
        "voice_evidence": voice_evidence,
        "decision": decision,
        "stage_status": {
            "intake": "ok", "stt": "ok", "text_analysis": "ok",
            "voice_evidence": "ok" if voice_evidence else "skipped", "triage": "ok",
        },
        # Synthetic timings verify shape only; they are not measured latency.
        "stage_timing_ms": {"triage": 0.0},
        "processing_ms": 0.0,
        "created_at": "2026-11-15T00:00:00Z",
    }

print(json.dumps({
    "python": platform.python_version(), "pydantic": pydantic.__version__,
    "branch": branch, "integration_ref_at_run": ref_commit,
    "contract_commit_used": CONTRACT_COMMIT,
    "schema_version": schemas.SCHEMA_VERSION, "label_version": schemas.LABEL_VERSION,
    "category_count": len(ComplaintCategory), "official_source_sha256": SOURCE_MANIFEST,
}, indent=2))


{
  "python": "3.12.10",
  "pydantic": "2.14.3",
  "branch": "team/ibthal-triage-routing",
  "integration_ref_at_run": "84205e98cc8d78fe9e949098c18e6a3ad27e96db",
  "contract_commit_used": "6f21d9bd20af4a175fd9ac011ac89384f32e4c97",
  "schema_version": 1,
  "label_version": "taxonomy-2026-11-15",
  "category_count": 11,
  "official_source_sha256": {
    "src/svr/schemas.py": "c70ade95f28ff78d697cccf20e3f14b51d0ed92c998085629d01349b7ff06615",
    "src/svr/agents.py": "0755a39e25dbd48b107fa20a820d3b97adc8222a72749deb154bfa503674819a",
    "src/svr/fixtures.py": "8046cf41014afd691c6cec508b16d1307650fa04ea408fb639482aeb446d54d3",
    "src/svr/stages/triage.py": "98b467468e1ec1092a82f0805aed27f02d7b8b2922a56e5744bcc24d691d0f48",
    "src/svr/stages/ticket.py": "904e1db4af714c937bdf5cb4ffe33b7db621801e9f470cc0307341e8bee8c810",
    "src/svr/latency.py": "c0ed4f344fd8242d0f777e8e19edf24883735ef098a87267dd262e7b8124e269",
    "src/svr/stages/intake.py": "41c70c5bc7513a6ff45ea609c8b40abe2af8862

## 2. Priority decision table

I add urgency points (high 3, medium 2, low 1), churn points (at_risk 2, not_at_risk 0), and customer_state_text points (mad/frustrated 1, calm/satisfied/unclear 0). Scores span 1 to 6. Buckets are 1-2 low, 3-4 normal, 5 high, and 6 urgent.

I preserve null priorities and human review when a required scoring input is missing. This missing-input policy requires Hazem confirmation. Invalid labels raise official validation errors before reservation. Voice evidence does not contribute.

| complaint_urgency | churn_risk | customer_state_text group | State points | priority_score | queue_priority |
|---|---|---|---|---|---|
| `high` | `at_risk` | mad/frustrated | 1 | 6 | `urgent` |
| `high` | `at_risk` | calm/satisfied/unclear | 0 | 5 | `high` |
| `high` | `not_at_risk` | mad/frustrated | 1 | 4 | `normal` |
| `high` | `not_at_risk` | calm/satisfied/unclear | 0 | 3 | `normal` |
| `medium` | `at_risk` | mad/frustrated | 1 | 5 | `high` |
| `medium` | `at_risk` | calm/satisfied/unclear | 0 | 4 | `normal` |
| `medium` | `not_at_risk` | mad/frustrated | 1 | 3 | `normal` |
| `medium` | `not_at_risk` | calm/satisfied/unclear | 0 | 2 | `low` |
| `low` | `at_risk` | mad/frustrated | 1 | 4 | `normal` |
| `low` | `at_risk` | calm/satisfied/unclear | 0 | 3 | `normal` |
| `low` | `not_at_risk` | mad/frustrated | 1 | 2 | `low` |
| `low` | `not_at_risk` | calm/satisfied/unclear | 0 | 1 | `low` |



In [2]:
PRIORITY_TABLE = []
for urgency, churn, state in product(ComplaintUrgency, ChurnRisk, [CustomerState.MAD, CustomerState.CALM]):
    score, bucket, _, _ = calculate_priority(synthetic_text(
        complaint_urgency=urgency, churn_risk=churn, customer_state_text=state))
    PRIORITY_TABLE.append({
        "complaint_urgency": urgency.value, "churn_risk": churn.value,
        "customer_state_group": "mad/frustrated = 1" if state == CustomerState.MAD else "calm/satisfied/unclear = 0",
        "priority_score": score, "queue_priority": bucket.value,
    })
assert len(PRIORITY_TABLE) == 12
print(json.dumps(PRIORITY_TABLE, indent=2))


[
  {
    "complaint_urgency": "high",
    "churn_risk": "at_risk",
    "customer_state_group": "mad/frustrated = 1",
    "priority_score": 6,
    "queue_priority": "urgent"
  },
  {
    "complaint_urgency": "high",
    "churn_risk": "at_risk",
    "customer_state_group": "calm/satisfied/unclear = 0",
    "priority_score": 5,
    "queue_priority": "high"
  },
  {
    "complaint_urgency": "high",
    "churn_risk": "not_at_risk",
    "customer_state_group": "mad/frustrated = 1",
    "priority_score": 4,
    "queue_priority": "normal"
  },
  {
    "complaint_urgency": "high",
    "churn_risk": "not_at_risk",
    "customer_state_group": "calm/satisfied/unclear = 0",
    "priority_score": 3,
    "queue_priority": "normal"
  },
  {
    "complaint_urgency": "medium",
    "churn_risk": "at_risk",
    "customer_state_group": "mad/frustrated = 1",
    "priority_score": 5,
    "queue_priority": "high"
  },
  {
    "complaint_urgency": "medium",
    "churn_risk": "at_risk",
    "customer_state_gro

## 3. Automated synthetic tests

I test assignment, unavailable agents, capacity limits and concurrent reservations, missing labels, contract validation, all 12 priority groups, all 11 schema categories, state/voice separation, repeatability, serialization, ticket handoff, and rejection paths. Each test restores the official fictional seeds. Expected validation exceptions count as successful checks.

I also verify that official store completion releases once and rejects a second completion without releasing again. I test input validation before reservation and rollback if decision construction fails. I make no claim about real routing accuracy or measured latency.


In [3]:
class Phase1TriageTests(unittest.TestCase):
    def setUp(self):
        self.original_seeds = copy.deepcopy(agents.SEEDED_AGENTS)
        agents.reset_loads()
        self.baseline = load_snapshot()

    def tearDown(self):
        # Off-shift/capacity scenarios change only private in-memory demo fixtures.
        agents.SEEDED_AGENTS[:] = self.original_seeds
        agents.reset_loads()

    def assert_decision(self, decision):
        TriageDecision.model_validate_json(decision.model_dump_json())
        self.assertEqual(set(decision.model_dump()), EXPECTED_DECISION_FIELDS)
        if decision.priority_score is None:
            self.assertIsNone(decision.queue_priority)
            self.assertIn("requires Hazem confirmation", decision.route_reason)
        else:
            self.assertIn(decision.priority_score, range(1, 7))
            expected = {1: "low", 2: "low", 3: "normal", 4: "normal", 5: "high", 6: "urgent"}
            self.assertEqual(decision.queue_priority.value, expected[decision.priority_score])
        self.assertIsNone(decision.evaluation)
        self.assertEqual(decision.config_version, CONFIG_VERSION)
        self.assertEqual(decision.schema_version, 1)
        self.assertEqual(decision.label_version, "taxonomy-2026-11-15")
        self.assertTrue(decision.route_reason.strip())
        self.assertEqual(
            decision.assignment_status == AssignmentStatus.ASSIGNED,
            decision.assigned_agent_id is not None,
        )

    def test_01_valid_assignment(self):
        text = synthetic_text()
        decision = triage_phase1(text)
        self.assert_decision(decision)
        agent = agents.get_agent(decision.assigned_agent_id)
        self.assertEqual(agent.id, "agent-demo-network")
        self.assertIn(text.complaint_category, agent.skills)
        self.assertTrue(agent.on_shift)
        self.assertLessEqual(agent.current_load, agent.capacity)
        self.assertEqual(agent.current_load, self.baseline[agent.id] + 1)
        self.assertIn("Fictional seeded demo", decision.route_reason)
        self.assertIn("customer_state_text=frustrated", decision.contributing_signals)

    def test_02_null_category(self):
        text = synthetic_text(complaint_category=None, needs_human_review=True)
        decision = triage_phase1(text)
        self.assert_decision(decision)
        self.assertEqual(decision.assignment_status, AssignmentStatus.UNASSIGNED)
        self.assertIn("category is missing", decision.route_reason)
        self.assertEqual(load_snapshot(), self.baseline)

    def test_03_unknown_category_rejected(self):
        with self.assertRaises(ValidationError):
            synthetic_text(complaint_category="unknown_category")
        self.assertEqual(load_snapshot(), self.baseline)

    def test_04_no_eligible_agent(self):
        agents.SEEDED_AGENTS[:] = []
        decision = triage_phase1(synthetic_text())
        self.assert_decision(decision)
        self.assertEqual(decision.assignment_status, AssignmentStatus.UNASSIGNED)
        self.assertIn("human routing", decision.route_reason)

    def test_05_off_shift(self):
        for agent in agents.SEEDED_AGENTS:
            agent.on_shift = False
        decision = triage_phase1(synthetic_text())
        self.assert_decision(decision)
        self.assertEqual(decision.assignment_status, AssignmentStatus.UNASSIGNED)
        self.assertEqual(load_snapshot(), self.baseline)

    def test_06_at_capacity(self):
        for agent in agents.SEEDED_AGENTS:
            agent.current_load = agent.capacity
        full = load_snapshot()
        decision = triage_phase1(synthetic_text())
        self.assert_decision(decision)
        self.assertEqual(decision.assignment_status, AssignmentStatus.UNASSIGNED)
        self.assertEqual(load_snapshot(), full)

    def test_07_reserve_once_release_once(self):
        decision = triage_phase1(synthetic_text())
        selected = decision.assigned_agent_id
        expected = dict(self.baseline)
        expected[selected] += 1
        self.assertEqual(load_snapshot(), expected)
        agents.release(selected)  # Standalone caller ends this synthetic run.
        self.assertEqual(load_snapshot(), self.baseline)

    def test_08_all_scoring_combinations(self):
        scores, buckets = set(), set()
        expected_buckets = {1: "low", 2: "low", 3: "normal", 4: "normal", 5: "high", 6: "urgent"}
        for urgency, churn, state in product(ComplaintUrgency, ChurnRisk, CustomerState):
            with self.subTest(urgency=urgency, churn=churn, state=state):
                agents.reset_loads()
                expected = {"high": 3, "medium": 2, "low": 1}[urgency.value]
                expected += 2 if churn.value == "at_risk" else 0
                expected += 1 if state.value in ("mad", "frustrated") else 0
                decision = triage_phase1(synthetic_text(complaint_urgency=urgency,
                    churn_risk=churn, customer_state_text=state))
                self.assert_decision(decision)
                self.assertEqual(decision.priority_score, expected)
                self.assertEqual(decision.queue_priority.value, expected_buckets[expected])
                self.assertIn(f"priority_score={expected}", decision.contributing_signals)
                self.assertIn(f"= {expected};", decision.route_reason)
                self.assertEqual(decision.assigned_agent_id, "agent-demo-network")
                scores.add(expected)
                buckets.add(decision.queue_priority.value)
        self.assertEqual(scores, set(range(1, 7)))
        self.assertEqual(buckets, {"low", "normal", "high", "urgent"})

    def test_09_missing_labels_preserve_review(self):
        for field in LABEL_FIELDS:
            with self.subTest(field=field):
                agents.reset_loads()
                with self.assertRaises(ValidationError):
                    synthetic_text(**{field: None}, needs_human_review=False)
                text = synthetic_text(**{field: None}, needs_human_review=True)
                before = text.model_dump_json()
                decision = triage_phase1(text)
                self.assert_decision(decision)
                if field != "complaint_category":
                    self.assertIsNone(decision.priority_score)
                    self.assertIsNone(decision.queue_priority)
                else:
                    self.assertIsNotNone(decision.priority_score)
                ticket = ticket_stage.assemble_ticket(**handoff_arguments(text, decision))
                self.assertTrue(ticket.needs_human_review)
                self.assertEqual(ticket.ticket_status, TicketStatus.NEEDS_REVIEW)
                self.assertIsNone(getattr(ticket, field))
                self.assertIn("still requires human review", decision.route_reason)
                self.assertEqual(text.model_dump_json(), before)
                if decision.assigned_agent_id:
                    agents.release(decision.assigned_agent_id)

    def test_10_explicit_review_with_complete_labels(self):
        text = synthetic_text(needs_human_review=True)
        decision = triage_phase1(text)
        ticket = ticket_stage.assemble_ticket(**handoff_arguments(text, decision))
        self.assertTrue(ticket.needs_human_review)
        self.assertEqual(ticket.ticket_status, TicketStatus.NEEDS_REVIEW)
        self.assertIn("needs_human_review=true", decision.contributing_signals)

    def test_11_invalid_enums(self):
        for field in LABEL_FIELDS:
            with self.subTest(field=field), self.assertRaises(ValidationError):
                synthetic_text(**{field: "invalid-label"})
        with self.assertRaises(ValidationError):
            synthetic_voice(voice_state_candidate="invalid-label")
        payload = triage_phase1(synthetic_text()).model_dump()
        for field in ("assignment_status", "queue_priority"):
            with self.subTest(field=field), self.assertRaises(ValidationError):
                TriageDecision.model_validate({**payload, field: "invalid-label"})

    def test_12_unexpected_fields(self):
        payloads = [
            (TextAnalysis, synthetic_text().model_dump()),
            (VoiceEvidence, synthetic_voice().model_dump()),
            (TriageDecision, triage_phase1(synthetic_text()).model_dump()),
        ]
        for model, payload in payloads:
            with self.subTest(model=model.__name__), self.assertRaises(ValidationError):
                model.model_validate({**payload, "unapproved_policy": True})

    def test_13_wrong_versions(self):
        payloads = [
            (TextAnalysis, synthetic_text().model_dump()),
            (VoiceEvidence, synthetic_voice().model_dump()),
            (TriageDecision, triage_phase1(synthetic_text()).model_dump()),
        ]
        for model, payload in payloads:
            for field, invalid in (("schema_version", 2), ("label_version", "invented-taxonomy")):
                with self.subTest(model=model.__name__, field=field):
                    with self.assertRaises(ValidationError):
                        model.model_validate({**payload, field: invalid})

    def test_14_state_voice_never_change_urgency(self):
        # Each text state crosses absent voice and all five voice states.
        for state, voice_state in product(CustomerState, [None, *CustomerState]):
            with self.subTest(state=state.value, voice_state=voice_state):
                agents.reset_loads()
                text = synthetic_text(customer_state_text=state, complaint_urgency="low")
                voice = synthetic_voice(voice_state_candidate=voice_state) if voice_state else None
                before = text.model_dump_json()
                voice_before = voice.model_dump_json() if voice else None
                decision = triage_phase1(text, voice)
                self.assert_decision(decision)
                self.assertEqual(decision.priority_score, 1 + int(state.value in ("mad", "frustrated")))
                self.assertEqual(decision.queue_priority.value, "low")
                self.assertIn("complaint_urgency=low", decision.contributing_signals)
                self.assertEqual(decision.assigned_agent_id, "agent-demo-network")
                ticket = ticket_stage.assemble_ticket(**handoff_arguments(text, decision, voice))
                self.assertEqual(ticket.complaint_urgency, ComplaintUrgency.LOW)
                self.assertEqual(ticket.customer_state_text, state)
                self.assertEqual(ticket.voice_state_candidate, voice_state)
                self.assertIsNone(ticket.customer_state_final)
                self.assertEqual(text.model_dump_json(), before)
                self.assertEqual(voice.model_dump_json() if voice else None, voice_before)

    def test_15_repeatability_after_reset(self):
        first = triage_phase1(synthetic_text(), synthetic_voice()).model_dump_json()
        agents.reset_loads()
        second = triage_phase1(synthetic_text(), synthetic_voice()).model_dump_json()
        self.assertEqual(first, second)

    def test_16_serialization_round_trip(self):
        decision = triage_phase1(synthetic_text())
        encoded = decision.model_dump_json()
        payload = json.loads(encoded)
        self.assertEqual(set(payload), EXPECTED_DECISION_FIELDS)
        self.assertEqual(payload["priority_score"], decision.priority_score)
        self.assertEqual(payload["queue_priority"], decision.queue_priority.value)
        self.assertEqual(payload["assignment_status"], "assigned")
        self.assertEqual(TriageDecision.model_validate_json(encoded), decision)
        self.assert_decision(decision)

    def test_17_ticket_assembly_handoff(self):
        text = synthetic_text()
        decision = triage_phase1(text)
        ticket = ticket_stage.assemble_ticket(**handoff_arguments(text, decision))
        self.assertEqual(ticket.assignment_status, decision.assignment_status)
        self.assertEqual(ticket.priority_score, decision.priority_score)
        self.assertEqual(ticket.route_reason, decision.route_reason)
        self.assertEqual(ticket.summary, text.summary)
        self.assertEqual(ticket.customer_request, text.customer_request)
        self.assertEqual(ticket.suggested_script, text.suggested_script)
        self.assertEqual(ticket.transcript, fixtures.FIXTURE_TRANSCRIPT)
        self.assertEqual(ticket.ticket_status, TicketStatus.COMPLETED)
        self.assertFalse(ticket.needs_human_review)
        self.assertFalse(ticket.cached)
        self.assertIsNone(ticket.customer_state_final)
        detail = schemas.TicketDetail(**ticket.model_dump(), triage_decision=decision)
        self.assertEqual(detail.triage_decision.assigned_agent_id, decision.assigned_agent_id)
        self.assertEqual(detail.triage_decision.queue_priority, decision.queue_priority)
        # Assembly copies a decision; it neither reserves nor releases capacity.
        self.assertEqual(agents.get_agent(decision.assigned_agent_id).current_load,
                         self.baseline[decision.assigned_agent_id] + 1)

    def test_18_assembler_rejects_mismatched_ids(self):
        text = synthetic_text()
        voice = synthetic_voice()
        decision = triage_phase1(text, voice)
        base = handoff_arguments(text, decision, voice)
        for field in ("audio", "text", "voice_evidence", "decision"):
            with self.subTest(field=field):
                changed = dict(base)
                payload = base[field].model_dump()
                payload["voicemail_id"] = "voicemail-other-synthetic"
                changed[field] = type(base[field]).model_validate(payload)
                with self.assertRaisesRegex(ValueError, "different voicemails"):
                    ticket_stage.assemble_ticket(**changed)
        with self.assertRaisesRegex(ValueError, "different voicemails"):
            ticket_stage.assemble_ticket(**{**base, "voicemail_id": "wrong-target"})
        # Failure does not release on its own; the owner of the reservation must.
        agents.release(decision.assigned_agent_id)
        self.assertEqual(load_snapshot(), self.baseline)

    def test_19_assembler_rejects_failed_stages(self):
        text = synthetic_text()
        decision = triage_phase1(text)
        base = handoff_arguments(text, decision)
        for stage in base["stage_status"]:
            with self.subTest(stage=stage):
                args = {**base, "stage_status": {**base["stage_status"], stage: "failed"}}
                with self.assertRaisesRegex(ValueError, "failed stages"):
                    ticket_stage.assemble_ticket(**args)
        agents.release(decision.assigned_agent_id)
        self.assertEqual(load_snapshot(), self.baseline)

    def test_20_triage_rejects_voice_id_before_reservation(self):
        with self.assertRaisesRegex(ValueError, "same voicemail"):
            triage_phase1(synthetic_text(), synthetic_voice(voicemail_id="other-synthetic"))
        self.assertEqual(load_snapshot(), self.baseline)

    def test_21_mutated_input_revalidated_before_reservation(self):
        text = synthetic_text()
        text.complaint_category = "unknown-category"
        with self.assertWarnsRegex(UserWarning, "Pydantic serializer warnings"):
            with self.assertRaises(ValidationError):
                triage_phase1(text)
        text = synthetic_text()
        text.customer_state_text = None
        with self.assertRaises(ValidationError):
            triage_phase1(text)
        voice = synthetic_voice()
        voice.schema_version = 2
        with self.assertRaises(ValidationError):
            triage_phase1(synthetic_text(), voice)
        self.assertEqual(load_snapshot(), self.baseline)

    def test_22_store_completion_releases_once(self):
        text = synthetic_text()
        decision = triage_phase1(text)
        args = handoff_arguments(text, decision)
        ticket = ticket_stage.assemble_ticket(**args)
        result = pipeline.PipelineResult(ticket, args["audio"], text, None, decision)
        store = store_module.TicketStore()
        self.assertEqual(agents.get_agent(decision.assigned_agent_id).current_load,
                         self.baseline[decision.assigned_agent_id] + 1)
        store.complete(result)  # Official lifecycle owner releases the reservation.
        self.assertEqual(load_snapshot(), self.baseline)
        detail = store.get_detail(ticket.id)
        self.assertEqual(detail.triage_decision, decision)
        with self.assertRaisesRegex(ValueError, "already finished"):
            store.complete(result)
        self.assertEqual(load_snapshot(), self.baseline)
        # I deliberately do not call release() after complete().

    def test_23_decision_failure_rolls_back_reservation(self):
        with patch.dict(triage_phase1.__globals__, {"TriageDecision": unittest.mock.Mock(
            side_effect=RuntimeError("synthetic construction failure")
        )}):
            with self.assertRaisesRegex(RuntimeError, "synthetic construction failure"):
                triage_phase1(synthetic_text())
        self.assertEqual(load_snapshot(), self.baseline)

    def test_24_all_eleven_schema_categories_use_existing_seed_skills(self):
        for category in ComplaintCategory:
            with self.subTest(category=category.value):
                agents.reset_loads()
                decision = triage_phase1(synthetic_text(complaint_category=category))
                self.assert_decision(decision)
                if category.value in ("offers_and_packages", "sim_services"):
                    self.assertIsNone(decision.assigned_agent_id)
                    self.assertEqual(load_snapshot(), self.baseline)
                    self.assertIn("human routing", decision.route_reason)
                else:
                    self.assertIn(category, agents.get_agent(decision.assigned_agent_id).skills)

    def test_25_concurrent_reservations_do_not_overbook(self):
        selected = agents.get_agent("agent-demo-network")
        remaining = selected.capacity - selected.current_load
        def route(index):
            return triage_phase1(synthetic_text(voicemail_id=f"synthetic-parallel-{index}"))
        with ThreadPoolExecutor(max_workers=8) as pool:
            decisions = list(pool.map(route, range(16)))
        assigned = [d for d in decisions if d.assignment_status == AssignmentStatus.ASSIGNED]
        self.assertEqual(len(assigned), remaining)
        self.assertEqual(agents.get_agent(selected.id).current_load, selected.capacity)
        for decision in decisions:
            self.assert_decision(decision)
        for decision in assigned:
            agents.release(decision.assigned_agent_id)
        self.assertEqual(load_snapshot(), self.baseline)

    def test_26_existing_mock_compatibility(self):
        expected = official_triage.triage_mock(synthetic_text(), None)
        agents.reset_loads()
        actual = triage_phase1(synthetic_text(), None)
        for field in ("voicemail_id", "assignment_status", "assigned_agent_id",
                      "schema_version", "label_version"):
            self.assertEqual(getattr(actual, field), getattr(expected, field))
        self.assert_decision(actual)

    def test_27_fusion_and_language_mix_remain_disabled(self):
        with self.assertRaises(ValidationError):
            synthetic_text(language_mix="invented-mix")
        text = synthetic_text()
        decision = triage_phase1(text)
        ticket = ticket_stage.assemble_ticket(**handoff_arguments(text, decision))
        with self.assertRaises(ValidationError):
            schemas.Ticket.model_validate({**ticket.model_dump(), "customer_state_final": "mad"})

    def test_28_tier_not_used_as_policy(self):
        for agent in agents.SEEDED_AGENTS:
            agent.tier = None
        decision = triage_phase1(synthetic_text())
        self.assertEqual(decision.assigned_agent_id, "agent-demo-network")
        self.assert_decision(decision)

    def test_29_input_types_and_voice_confidence(self):
        with self.assertRaises(TypeError):
            triage_phase1(synthetic_text().model_dump())
        with self.assertRaises(TypeError):
            triage_phase1(synthetic_text(), {})
        for value in (-0.01, 1.01):
            with self.subTest(confidence=value), self.assertRaises(ValidationError):
                synthetic_voice(voice_confidence=value)
        self.assertEqual(load_snapshot(), self.baseline)

    def test_30_exact_taxonomy_and_previous_version_rejected(self):
        expected = {"network_and_connectivity", "billing_and_charges", "balance_and_data",
                    "app_and_account", "branch_and_staff", "no_reply_or_long_wait",
                    "cancellation_request", "thanks_or_praise", "other", "offers_and_packages", "sim_services"}
        self.assertEqual({c.value for c in ComplaintCategory}, expected)
        with self.assertRaises(ValidationError):
            synthetic_text(complaint_category="greeting_only")
        for model, payload in (
            (TextAnalysis, synthetic_text().model_dump()),
            (VoiceEvidence, synthetic_voice().model_dump()),
            (TriageDecision, triage_phase1(synthetic_text()).model_dump()),
            (schemas.AudioEvidence, synthetic_audio("synthetic").model_dump()),
        ):
            with self.subTest(model=model.__name__), self.assertRaises(ValidationError):
                model.model_validate({**payload, "label_version": "taxonomy-2026-11-11"})

    def test_31_all_missing_scoring_combinations(self):
        fields = ("complaint_urgency", "churn_risk", "customer_state_text")
        for flags in product((False, True), repeat=3):
            if not any(flags):
                continue
            with self.subTest(missing=flags):
                agents.reset_loads()
                changes = {field: None for field, missing in zip(fields, flags) if missing}
                text = synthetic_text(**changes, needs_human_review=True)
                decision = triage_phase1(text, synthetic_voice(voice_state_candidate="mad"))
                self.assertIsNone(decision.priority_score)
                self.assertIsNone(decision.queue_priority)
                self.assertIn("requires Hazem confirmation", decision.route_reason)
                ticket = ticket_stage.assemble_ticket(**handoff_arguments(text, decision))
                self.assertTrue(ticket.needs_human_review)
                self.assertEqual(ticket.ticket_status, TicketStatus.NEEDS_REVIEW)

# I run all cases even after a failure, print the evidence, then fail the cell
# if any case fails or skips. Expected validation errors are passing tests.
suite = unittest.defaultTestLoader.loadTestsFromTestCase(Phase1TriageTests)
test_log = io.StringIO()
result = unittest.TextTestRunner(stream=test_log, verbosity=2).run(suite)
print(test_log.getvalue())
TEST_SUMMARY = {
    "test_methods": result.testsRun,
    "passed": result.testsRun - len(result.failures) - len(result.errors) - len(result.skipped),
    "failed": len(result.failures),
    "errors": len(result.errors),
    "skipped": len(result.skipped),
    "scope": "synthetic fixtures only; no routing-agreement or latency claim",
}
print(json.dumps(TEST_SUMMARY, indent=2))
assert result.wasSuccessful() and not result.skipped, "Inspect the test log above."
assert load_snapshot() == INITIAL_LOADS, "Tests must restore the seeded loads."


test_01_valid_assignment (__main__.Phase1TriageTests.test_01_valid_assignment) ... ok
test_02_null_category (__main__.Phase1TriageTests.test_02_null_category) ... ok
test_03_unknown_category_rejected (__main__.Phase1TriageTests.test_03_unknown_category_rejected) ... ok
test_04_no_eligible_agent (__main__.Phase1TriageTests.test_04_no_eligible_agent) ... ok
test_05_off_shift (__main__.Phase1TriageTests.test_05_off_shift) ... ok
test_06_at_capacity (__main__.Phase1TriageTests.test_06_at_capacity) ... ok
test_07_reserve_once_release_once (__main__.Phase1TriageTests.test_07_reserve_once_release_once) ... ok
test_08_all_scoring_combinations (__main__.Phase1TriageTests.test_08_all_scoring_combinations) ... ok
test_09_missing_labels_preserve_review (__main__.Phase1TriageTests.test_09_missing_labels_preserve_review) ... ok
test_10_explicit_review_with_complete_labels (__main__.Phase1TriageTests.test_10_explicit_review_with_complete_labels) ... ok
test_11_invalid_enums (__main__.Phase1TriageTest

## 4. Ticket handoff and capacity lifecycle

I pass my decision to Arafa's unchanged `assemble_ticket` function. It copies routing fields, takes the summary/request/script from text analysis, takes the transcript from synthetic audio evidence, and derives review status. The base Ticket does not include `queue_priority` or `assigned_agent_id`; `TicketDetail.triage_decision` retains the full decision.

I then pass the assembled result to the official `TicketStore.complete`. The printed load snapshots show one reservation and one release. Synthetic timing values exercise the contract shape only.


In [4]:
# I reserve for one synthetic run, assemble through Arafa, and let the
# official store complete the run and release the reservation exactly once.
agents.reset_loads()
loads_before = load_snapshot()
text = synthetic_text()
decision = triage_phase1(text, synthetic_voice())
args = handoff_arguments(text, decision, synthetic_voice())
ticket = ticket_stage.assemble_ticket(**args)
loads_reserved = load_snapshot()
demo_store = store_module.TicketStore()
demo_store.complete(
    pipeline.PipelineResult(ticket, args["audio"], text, args["voice_evidence"], decision)
)
loads_after = load_snapshot()
assert loads_after == loads_before
assert loads_reserved[decision.assigned_agent_id] == loads_before[decision.assigned_agent_id] + 1
assert ticket.priority_score == decision.priority_score == 3
assert decision.queue_priority == schemas.QueuePriority.NORMAL
assert ticket.customer_state_final is None
print(json.dumps({
    "decision": decision.model_dump(mode="json"),
    "ticket_status": ticket.ticket_status.value,
    "needs_human_review": ticket.needs_human_review,
    "capacity_before": loads_before, "capacity_reserved": loads_reserved,
    "capacity_after_store_completion": loads_after,
}, indent=2))


{
  "decision": {
    "voicemail_id": "voicemail-fixture-001",
    "priority_score": 3.0,
    "queue_priority": "normal",
    "assigned_agent_id": "agent-demo-network",
    "assignment_status": "assigned",
    "route_reason": "Fictional seeded demo routing assigns agent-demo-network: its listed skill matches network_and_connectivity, it is on shift, and one capacity unit is reserved. Selection uses the existing demo seed order, not an approved production tie-breaking policy. Priority calculation: 2 + 0 + 1 = 3; queue_priority=normal. Voice fusion is disabled.",
    "contributing_signals": [
      "complaint_category=network_and_connectivity",
      "complaint_urgency=medium",
      "customer_state_text=frustrated",
      "churn_risk=not_at_risk",
      "needs_human_review=false",
      "voice_evidence=ignored_fusion_disabled",
      "urgency_points=2",
      "churn_points=0",
      "customer_state_points=1",
      "priority_score=3",
      "queue_priority=normal"
    ],
    "config_ver

## 5. Findings and remaining decisions

I verify 11 categories at commit `6f21d9bd20af4a175fd9ac011ac89384f32e4c97`, schema version `1`, label version `taxonomy-2026-11-15`. The diff from `84205e98cc8d78fe9e949098c18e6a3ad27e96db` removes greeting_only and adds offers_and_packages and sim_services. General-agent skills lose greeting_only. The two new categories have no seeded skills and stay unassigned for human routing. The assembler, store, pipeline, mock triage, and fixture interfaces have identical Git blobs across these revisions.

I implement Hazem's approved priority formula and keep fusion disabled. Missing scoring signals retain null priorities with human review, pending Hazem confirmation of this missing-input policy. Invalid inputs raise validation errors before capacity reservation.

I preserve demo seed order, atomic reservation, rollback, and official ticket assembly. Production skills, tier policy, tie-breaking, fusion, and language_mix values remain pending. Shared schema comments and integration README still describe priority as open, so Arafa needs to align those descriptions during integration.

Assignment and review remain independent. Complete labels with an unsupported seed skill produce an unassigned ticket, while the current assembler can still mark it completed. Production integration must define that handoff, use shared runtime agent state, and settle retry/error cleanup ownership. Retrying triage reserves again. A standalone release must occur once, and store completion owns release when used.

The saved synthetic tests verify contract compatibility and lifecycle behavior. Deployment, full API/reassignment testing, model evaluation, and the 80 percent routing agreement target remain later work.
